<a href="https://colab.research.google.com/github/ugurcanselvi/Apartman-ve-Aidat-Y-netim-Sistemi/blob/main/Programlama_2_Proje.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import json
import os
from abc import ABC, abstractmethod

# 1. Abstract Class ve Encapsulation
class Kisi(ABC):
    def __init__(self, ad, soyad):
        self.ad = ad
        self.soyad = soyad

    @abstractmethod
    def bilgi_goster(self):
        """Kişi bilgilerini gösteren soyut metot."""
        pass

# 2. Kalıtım (Inheritance) ve Kapsülleme (Encapsulation)
class ApartmanSakini(Kisi):
    def __init__(self, ad, soyad, tc_no, telefon):
        super().__init__(ad, soyad)
        # Kapsüllenmiş nitelikler (private members)
        self.__tc_no = tc_no
        self.__telefon = telefon

    # Kapsüllenmiş değerlere erişim için property (Getter / Setter)
    @property
    def tc_no(self):
        return self.__tc_no

    @tc_no.setter
    def tc_no(self, value):
        self.__tc_no = value

    @property
    def telefon(self):
        return self.__telefon

    @telefon.setter
    def telefon(self, value):
        self.__telefon = value

    # 3. Çok Biçimlilik (Polymorphism) - Override işlemi
    def bilgi_goster(self):
        return f"Sakin: {self.ad} {self.soyad} | Tel: {self.__telefon}"

    def to_dict(self):
        return {
            "tip": "sakin",
            "ad": self.ad,
            "soyad": self.soyad,
            "tc_no": self.__tc_no,
            "telefon": self.__telefon
        }

class Yonetici(Kisi):
    def __init__(self, ad, soyad, gorev_baslangici):
        super().__init__(ad, soyad)
        self.gorev_baslangici = gorev_baslangici

    # Polymorphism
    def bilgi_goster(self):
        return f"Yönetici: {self.ad} {self.soyad} | Görev: {self.gorev_baslangici}"

class Aidat:
    def __init__(self, miktar, ay, yil, odenme_durumu=False):
        self.miktar = miktar
        self.ay = ay
        self.yil = yil
        self.odenme_durumu = odenme_durumu

    def odeme_yap(self):
        self.odenme_durumu = True

    def to_dict(self):
        return {
            "miktar": self.miktar,
            "ay": self.ay,
            "yil": self.yil,
            "odenme_durumu": self.odenme_durumu
        }

class Daire:
    def __init__(self, daire_no):
        self.daire_no = daire_no
        self.sakin = None
        self.aidatlar = []  # Liste kullanımı

    def sakin_ekle(self, sakin: ApartmanSakini):
        self.sakin = sakin

    def aidat_ekle(self, aidat: Aidat):
        self.aidatlar.append(aidat)

    def toplam_borc(self):
        borc = 0
        for aidat in self.aidatlar:
            if not aidat.odenme_durumu:
                borc += aidat.miktar
        return borc

    def to_dict(self):
        return {
            "daire_no": self.daire_no,
            "sakin": self.sakin.to_dict() if self.sakin else None,
            "aidatlar": [a.to_dict() for a in self.aidatlar]
        }

class ApartmanSistemi:
    def __init__(self, kayit_dosyasi="apartman_verileri.json"):
        self.daireler = {}  # Sözlük kullanımı {daire_no: Daire}
        self.kayit_dosyasi = kayit_dosyasi

    def daire_ekle(self, daire_no):
        if daire_no in self.daireler:
            print(f"[-] Hata: {daire_no} numaralı daire zaten sistemde mevcut.")
            return False
        self.daireler[daire_no] = Daire(daire_no)
        print(f"[+] {daire_no} numaralı daire başarıyla eklendi.")
        return True

    def sakin_ekle(self, daire_no, ad, soyad, tc_no, telefon):
        try:
            if daire_no not in self.daireler:
                print(f"[-] Hata: {daire_no} numaralı daire bulunamadı. Önce daireyi ekleyin.")
                return False

            yeni_sakin = ApartmanSakini(ad, soyad, tc_no, telefon)
            self.daireler[daire_no].sakin_ekle(yeni_sakin)
            print(f"[+] {ad} {soyad}, {daire_no} numaralı daireye kaydedildi.")
            return True
        except Exception as e:
            print(f"[-] Sakin eklenirken hata oluştu: {e}")
            return False

    def aidat_olustur(self, miktar, ay, yil):
        try:
            eklenen_sayisi = 0
            for daire in self.daireler.values():
                if daire.sakin is not None:
                    yeni_aidat = Aidat(miktar, ay, yil)
                    daire.aidat_ekle(yeni_aidat)
                    eklenen_sayisi += 1
            print(f"[+] Başarılı: {eklenen_sayisi} dolu daireye {ay}/{yil} dönemi için {miktar} TL aidat yansıtıldı.")
        except Exception as e:
            print(f"[-] Aidat oluştururken hata: {e}")

    def aidat_odeme(self, daire_no, ay, yil):
        try:
            if daire_no not in self.daireler:
                print("[-] Daire bulunamadı!")
                return

            daire = self.daireler[daire_no]
            odeme_yapildi = False
            for aidat in daire.aidatlar:
                if str(aidat.ay) == str(ay) and str(aidat.yil) == str(yil) and not aidat.odenme_durumu:
                    aidat.odeme_yap()
                    odeme_yapildi = True
                    print(f"[+] {daire_no} nolu dairenin {ay}/{yil} dönemi aidatı ({aidat.miktar} TL) ödendi.")
                    break

            if not odeme_yapildi:
                print("[-] Belirtilen döneme ait ödenmemiş aidat bulunamadı veya zaten ödenmiş.")
        except Exception as e:
            print(f"[-] Ödeme sırasında hata oluştu: {e}")

    def borc_sorgula(self, daire_no):
        if daire_no in self.daireler:
            toplam = self.daireler[daire_no].toplam_borc()
            print(f"[i] {daire_no} nolu dairenin toplam borcu: {toplam} TL")
            return toplam
        else:
            print("[-] Daire bulunamadı.")
            return 0

    def aidat_raporu_olustur(self):
        print("\n--- AİDAT RAPORU ---")
        if not self.daireler:
            print("Sistemde kayıtlı daire yok.")
        for d_no, daire in self.daireler.items():
            sakin_bilgisi = daire.sakin.bilgi_goster() if daire.sakin else "Boş Daire"
            toplam_borc = daire.toplam_borc()
            print(f"Daire {d_no} | {sakin_bilgisi} | Toplam Borç: {toplam_borc} TL")
        print("--------------------\n")

    def daireleri_listele(self):
        print("\n--- DAİRELER LİSTESİ ---")
        if not self.daireler:
            print("Sistemde henüz kayıtlı daire yok.")
        for d_no, daire in self.daireler.items():
            durum = "Dolu" if daire.sakin else "Boş"
            sakin_ad = f"{daire.sakin.ad} {daire.sakin.soyad}" if daire.sakin else "-"
            print(f"Daire: {d_no} | Durum: {durum} | Sakin: {sakin_ad}")
        print("------------------------\n")

    def geciken_odemeleri_goruntule(self):
        print("\n--- GECİKEN ÖDEMELER ---")
        geciken_var_mi = False
        for d_no, daire in self.daireler.items():
            if daire.sakin:
                for aidat in daire.aidatlar:
                    if not aidat.odenme_durumu:
                        print(f"Daire: {d_no} | Sakin: {daire.sakin.ad} {daire.sakin.soyad} | Dönem: {aidat.ay}/{aidat.yil} | Tutar: {aidat.miktar} TL")
                        geciken_var_mi = True
        if not geciken_var_mi:
            print("[+] Sistemde geciken ödeme bulunmamaktadır.")
        print("------------------------\n")

    def verileri_kaydet(self):
        try:
            # 4. Dosya İşlemleri (JSON formatında kayıt)
            with open(self.kayit_dosyasi, 'w', encoding='utf-8') as dosya:
                veri = {
                    "daireler": {k: v.to_dict() for k, v in self.daireler.items()}
                }
                json.dump(veri, dosya, ensure_ascii=False, indent=4)
            print(f"[+] Veriler başarıyla '{self.kayit_dosyasi}' dosyasına kaydedildi.")
        except Exception as e:
            print(f"[-] Kayıt işlemi sırasında hata oluştu: {e}")

    def verileri_yukle(self):
        try:
            if not os.path.exists(self.kayit_dosyasi):
                print("[i] Kayıt dosyası bulunamadı. Yeni ve boş bir sistem başlatılıyor.")
                return

            with open(self.kayit_dosyasi, 'r', encoding='utf-8') as dosya:
                veri = json.load(dosya)

            self.daireler.clear()
            for d_no, d_veri in veri.get("daireler", {}).items():
                yeni_daire = Daire(d_veri["daire_no"])

                s_veri = d_veri.get("sakin")
                if s_veri:
                    sakin = ApartmanSakini(s_veri["ad"], s_veri["soyad"], s_veri["tc_no"], s_veri["telefon"])
                    yeni_daire.sakin_ekle(sakin)

                for a_veri in d_veri.get("aidatlar", []):
                    aidat = Aidat(a_veri["miktar"], a_veri["ay"], a_veri["yil"], a_veri["odenme_durumu"])
                    yeni_daire.aidat_ekle(aidat)

                self.daireler[d_no] = yeni_daire

            print(f"[+] Veriler başarıyla yüklendi! Toplam {len(self.daireler)} daire bulundu.")
        except Exception as e:
            print(f"[-] Veri yükleme sırasında hata oluştu: {e}")
            import sys

def menu_goster():
    print("\n" + "="*35)
    print("      APARTMAN YÖNETİM SİSTEMİ      ")
    print("="*35)
    print("1- Daire ekle")
    print("2- Apartman sakini ekle")
    print("3- Aidat oluştur")
    print("4- Aidat ödeme")
    print("5- Borç sorgula")
    print("6- Aidat raporu oluştur")
    print("7- Daireleri listele")
    print("8- Geciken ödemeleri görüntüle")
    print("9- Verileri kaydet")
    print("10- Verileri yükle")
    print("0- Çıkış")
    print("="*35)

def main():
    sistem = ApartmanSistemi(kayit_dosyasi="apartman_verileri.json")

    print("\nSistem başlatılıyor...")
    sistem.verileri_yukle()

    while True:
        menu_goster()
        secim = input("Lütfen yapmak istediğiniz işlemi seçin (0-10): ")

        if secim == "1":
            d_no = input("Eklenecek Daire No (örn: 5): ")
            sistem.daire_ekle(d_no)

        elif secim == "2":
            d_no = input("Sakin Eklenecek Daire No: ")
            ad = input("Ad: ")
            soyad = input("Soyad: ")
            tc_no = input("TC No: ")
            tel = input("Telefon: ")
            sistem.sakin_ekle(d_no, ad, soyad, tc_no, tel)

        elif secim == "3":
            try:
                # Try-except ile hata yönetimi
                miktar = float(input("Aidat Miktarı (TL): "))
                ay = input("Ay (örn: 01 veya Ocak): ")
                yil = input("Yıl (örn: 2024): ")
                sistem.aidat_olustur(miktar, ay, yil)
            except ValueError:
                print("[-] Hata: Lütfen aidat miktarını sayısal bir değer olarak girin!")

        elif secim == "4":
            d_no = input("Ödeme Yapacak Daire No: ")
            ay = input("Ödenecek Ay: ")
            yil = input("Ödenecek Yıl: ")
            sistem.aidat_odeme(d_no, ay, yil)

        elif secim == "5":
            d_no = input("Borcu Sorgulanacak Daire No: ")
            sistem.borc_sorgula(d_no)

        elif secim == "6":
            sistem.aidat_raporu_olustur()

        elif secim == "7":
            sistem.daireleri_listele()

        elif secim == "8":
            sistem.geciken_odemeleri_goruntule()

        elif secim == "9":
            sistem.verileri_kaydet()

        elif secim == "10":
            sistem.verileri_yukle()

        elif secim == "0":
            try:
                onay = input("Çıkmadan önce veriler kaydedilsin mi? (E/H): ").lower()
                if onay == 'e':
                    sistem.verileri_kaydet()
                print("Çıkış yapılıyor... İyi günler!")
                sys.exit(0)
            except Exception as e:
                sys.exit(0)

        else:
            print("[-] Geçersiz seçim! Lütfen 0 ile 10 arasında bir rakam girin.")

if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n[!] Program kullanıcı tarafından zorla sonlandırıldı.")
        sys.exit(1)
    except Exception as e:
        print(f"\n[-] Beklenmeyen bir hata oluştu: {e}")


Sistem başlatılıyor...
[+] Veriler başarıyla yüklendi! Toplam 2 daire bulundu.

      APARTMAN YÖNETİM SİSTEMİ      
1- Daire ekle
2- Apartman sakini ekle
3- Aidat oluştur
4- Aidat ödeme
5- Borç sorgula
6- Aidat raporu oluştur
7- Daireleri listele
8- Geciken ödemeleri görüntüle
9- Verileri kaydet
10- Verileri yükle
0- Çıkış
Lütfen yapmak istediğiniz işlemi seçin (0-10): 0
Çıkmadan önce veriler kaydedilsin mi? (E/H): E
[+] Veriler başarıyla 'apartman_verileri.json' dosyasına kaydedildi.
Çıkış yapılıyor... İyi günler!


SystemExit: 0

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
